# Keep what you need

**Rung 2 of 5 (next: rung 3, [05, Change and rerun](05_change_and_rerun.ipynb)). You are here: the
same capture, less of it.**
Trigger: memory or time ran out.

A full trace saves every activation, which is fine for a toy model and impossible for a real one.
This notebook adds two arguments to the call you already type: `save=` chooses which sites keep
their tensors and `halt=` ends the forward pass once you have them. Then come the one-line
doors for when all you want is tensors (`tl.pluck`, `tl.extract`, `tl.extract_dataset`), and
saving a trace to disk so you never run the model twice for the same numbers.

> **You'll want to know:** [labels and module addresses](02_address_and_read.ipynb) (`relu_1_2`,
> `blocks.0`), [reading a record](02_address_and_read.ipynb) (`trace["..."].out`, `.shape`,
> `.parents`), [the summary](01_capture_and_look.ipynb) (what a full trace costs).

In [ ]:
import tempfile
from pathlib import Path

import torch
from IPython.display import display

import torchlens as tl

PICTURES = Path(
    tempfile.mkdtemp(prefix="torchlens_learn_")
)  # drawings land here, not beside the notebook

In [ ]:
from learn_models import (
    tiny_cnn,
    tiny_dataset,
    tiny_mlp,
)  # tiny seeded CPU models; nothing downloads

cnn, image = (
    tiny_cnn()
)  # six conv blocks, each conv, BatchNorm, ReLU, and a (2, 1, 16, 16) image batch

Same call, one more argument: `save=` keeps the tensors at the sites you name and drops the rest.

In [ ]:
everything = tl.trace(cnn, image)
relus_only = tl.trace(cnn, image, save=tl.func("relu"))
print(
    f"everything : {len(everything.saved_ops):2d} ops saved, {everything.cost.saved_activation_memory}"
)
print(
    f"relus only : {len(relus_only.saved_ops):2d} ops saved, {relus_only.cost.saved_activation_memory}"
)

## 1. One keyword: save=

`save=` takes a list of addresses (labels or module names), a selector, or `"none"`. Whatever you
drop, the metadata stays: every op still knows its shape, dtype, parents and children, and the
model's output is always kept. Only the tensors go, and a dropped activation reads as `None`.
One small thing is always kept as well: BatchNorm's running statistics, which appear as `buffer_*`
ops; the lists below hide them to stay short.

In [ ]:
def saved(trace):
    """Labels of the saved ops, without BatchNorm's running-statistic buffers."""
    return [label for label in trace.saved_ops.keys() if not label.startswith("buffer")]


two_sites = tl.trace(cnn, image, save=["relu_1_2", "head"])  # a label and a module name
print("saved:", saved(two_sites))

metadata_only = tl.trace(cnn, image, save="none")
relu = metadata_only["relu_1_2"]
print(
    "still known:",
    tuple(relu.shape),
    relu.dtype,
    "| parents",
    relu.parents,
    "| children",
    relu.children,
)
print("activation:", relu.out)

## 2. Predicates compose

A selector is a small object you can combine: `&` means both, `|` means either, `~` means not.
`tl.func` picks by the function that ran, `tl.in_module` by the module that owned the call,
`tl.label_contains` by the label text. `trace[selector]` resolves a selector against a finished
trace and returns a `Selection`, so you can count what a selector would keep before you commit
a capture to it.

In [ ]:
selectors = {
    "relus in block 3": tl.func("relu") & tl.in_module("blocks.3"),
    "convs or linear": tl.func("conv2d") | tl.func("linear"),
    "outside the blocks": ~tl.in_module("blocks"),
    "label contains relu": tl.label_contains("relu"),
}
for name, selector in selectors.items():
    selection = everything[selector]
    print(f"{name:>20}: {len(selection):2d} sites, first {list(selection.labels[:3])}")

**Optional detour.** Two selectors look along the op stream instead of at one op.
`tl.followed_by(s)` keeps an op whose next consumer matches `s`; `tl.preceded_by(s)` keeps an op
whose producer matched `s`. Both only make sense while the forward runs, so they go in `save=`
rather than in `trace[...]`, and they need a window of recent ops kept with their values to look
through. That window is two more arguments to `tl.trace`: `lookback=` sets its length and
`lookback_payload_policy=` keeps the values. In this CNN only the stem's conv feeds
a ReLU directly; inside the blocks a BatchNorm sits between. Skip to section 3 if you do not need
this yet.


In [ ]:
conv_then_relu = tl.func("conv2d") & tl.followed_by(tl.func("relu"))
direct = tl.trace(
    cnn, image, save=conv_then_relu, lookback=4, lookback_payload_policy="detached_raw"
)
print("convs feeding a relu directly:", saved(direct))

after_norm = tl.func("relu") & tl.preceded_by(tl.func("batch_norm"))
normed = tl.trace(cnn, image, save=after_norm, lookback=4, lookback_payload_policy="detached_raw")
print("relus right after a BatchNorm:", saved(normed))

## 3. Stop early

If the sites you want all sit in the first block, the other five blocks are wasted compute.
`halt=` ends the forward pass once a site matching the selector has run; the trace says so in its
outcome. `tl.module(...)` names a module, `tl.func(...)` a function.

In [ ]:
early = tl.trace(cnn, image, save=tl.in_module("blocks.0"), halt=tl.module("blocks.0"))
print(f"{len(early.ops)} ops ran of {len(everything.ops)} | {early.outcome.status.name}")
print("block 0 output:", tuple(early["blocks.0"].out.shape))

first_relu = tl.trace(cnn, image, save="relu_1_2", halt=tl.func("relu"))
print(
    f"{len(first_relu.ops)} ops ran | {first_relu.outcome.status.name} | saved {saved(first_relu)}"
)

## 4. The one-liners

When the tensor is all you want, skip the `Trace`. `tl.pluck` returns one activation;
`tl.extract` returns a dict of them. Both run the same capture underneath and save only what you
asked for; `tl.pluck` also takes `stop_after=` (its spelling of `halt=`), so the forward can end at the site, as it did in
section 3.


In [ ]:
block0 = tl.pluck(cnn, image, "blocks.0")
early_block0 = tl.pluck(cnn, image, "blocks.0", stop_after="blocks.0")  # and stop there
print(
    type(block0).__name__,
    tuple(block0.shape),
    "| same tensor when stopped early:",
    torch.equal(block0, early_block0),
)

acts = tl.extract(cnn, image, {"early": "blocks.0", "late": "blocks.5"})
{name: tuple(act.shape) for name, act in acts.items()}

## 5. Over a dataset

`tl.extract_dataset` runs the same extraction over many inputs in batches and stacks the results,
so the first axis is the item axis. Give it a `path=` and it writes shards to disk as it goes, with
a manifest describing what was extracted and how; `tl.load` reads the folder back.

In [ ]:
model, _ = tiny_mlp()  # the two-layer MLP from the quick-start
data = tiny_dataset(64)  # 64 inputs for it, as one (64, 8) tensor
acts = tl.extract_dataset(model, data, ["relu_1_2", "linear_2_3"], batch_size=16, progress=False)
{name: tuple(act.shape) for name, act in acts.items()}

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    folder = Path(tmp) / "acts"
    shards = tl.extract_dataset(
        model, data, ["relu_1_2"], batch_size=16, path=folder, progress=False
    )
    loaded = tl.load(folder)
    print(len(shards), "shards on disk; loaded as a", type(loaded).__name__)
    print("manifest keys:", sorted(loaded.manifest)[:6], "...")
    print("relu_1_2:", tuple(loaded.activations["relu_1_2"].shape))

## 6. Save a trace, load it back

A whole trace saves to a `.tlspec` folder and loads back as a `Trace`, saved tensors and graph
included. What the save kept, the loaded trace answers exactly: here the ReLU tensor and the
graph round-trip, so an expensive capture is run once and read many times. The model's output
is an op like any other: this trace saved only the ReLUs, so rebuilding the output refuses with a
typed error; add the output op to `save=` when you want it back.


In [ ]:
import warnings

with tempfile.TemporaryDirectory() as tmp, warnings.catch_warnings():
    warnings.simplefilter(
        "ignore"
    )  # the save lists what it embeds verbatim (BatchNorm buffers); fine for a toy
    path = Path(tmp) / "relus_only.tlspec"
    relus_only.save(path, overwrite=True)
    loaded = tl.load(path)
    print(
        type(loaded).__name__,
        "|",
        len(loaded.saved_ops),
        "ops saved |",
        sorted(p.name for p in path.iterdir())[:4],
    )
    print("same tensor:", torch.equal(relus_only["relu_1_2"].out, loaded["relu_1_2"].out))
    print("same graph :", len(loaded.ops) == len(relus_only.ops), "|", loaded["relu_1_2"].parents)
    try:
        loaded.reconstruct_output()
    except tl.errors.PayloadUnavailableError as error:
        print("model output after load:", str(error).split(";")[0])

## 7. That was rung 2

Two arguments and three one-liners: `save=` chooses sites by address or by selector, `halt=`
ends the pass, `tl.pluck` and `tl.extract` give tensors without a `Trace`, `tl.extract_dataset`
does it over a dataset, and `trace.save` with `tl.load` keeps any of it for later.

**One thing about networks:** every block in this CNN keeps the full 16 by 16 resolution, so each
stores the same activation volume, `(2, 8, 16, 16)`, and the six repeated blocks hold most of the
parameters (3.6K of 3,770). In a network that downsamples, the early layers hold most of the
activation memory and the late ones most of the parameters; `save=` is how you pay for only the
part you need.

**Go deeper:** next is [05, Change and rerun](05_change_and_rerun.ipynb): the selector you just
typed chooses what to edit. The three-line cards for this rung are
[extract_activations](../../examples/5min/extract_activations.ipynb),
[extract_dataset](../../examples/5min/extract_dataset.ipynb),
[pluck](../../examples/5min/pluck.ipynb) and [save_load](../../examples/5min/save_load.ipynb); the
workflow is [cog_neuro_extraction](../../examples/50min/cog_neuro_extraction.ipynb). Reference:
the glossary's "Site selectors", "Extraction, observers, and admin" and "Save / load" sections in
[glossary.md](../../docs/reference/glossary.md).